In [1]:
import os
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.model_selection import train_test_split

from torch.optim.lr_scheduler import CosineAnnealingLR

In [2]:
data_path = '/kaggle/input/competitions/cassava-leaf-disease-classification'

df = pd.read_csv(os.path.join(data_path, 'train.csv'))
df.head()

,image_id,label
0,1000015157.jpg,0
1,1000201771.jpg,3
2,100042118.jpg,1
3,1000723321.jpg,1
4,1000812911.jpg,3


In [3]:
train_df, valid_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)

print("train size:", len(train_df))
print("valid size:", len(valid_df))

train size: 17117
valid size: 4280


In [4]:
train_transform=transforms.Compose([
transforms.RandomResizedCrop(224,scale=(0.8,1.0)),
transforms.RandomHorizontalFlip(p=0.5),
transforms.ToTensor(),
transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

valid_transform=transforms.Compose([
transforms.Resize((224,224)),
transforms.ToTensor(),
transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

In [5]:
import torch
print("torch version:", torch.__version__)
print("torch cuda version:", torch.version.cuda)
print("gpu name:", torch.cuda.get_device_name(0))
print("gpu capability:", torch.cuda.get_device_capability(0))

torch version: 2.10.0+cu128
torch cuda version: 12.8
gpu name: Tesla T4
gpu capability: (7, 5)


In [6]:
class CassavaDataset(Dataset):
    def __init__(self, df, image_dir, transform=None):
        self.df = df
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image_id = self.df.loc[idx, 'image_id']
        label = self.df.loc[idx, 'label']

        img_path = os.path.join(self.image_dir, image_id)
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, label

In [7]:
train_img_dir = os.path.join(data_path, 'train_images')

train_dataset = CassavaDataset(train_df, train_img_dir, transform=train_transform)
valid_dataset = CassavaDataset(valid_df, train_img_dir, transform=valid_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False)

In [8]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, 5)
model = model.to(device)

cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 183MB/s] 


In [9]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = CosineAnnealingLR(optimizer, T_max=10)

In [10]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    
    return epoch_loss, epoch_acc

In [11]:
def valid_one_epoch(model, loader, criterion, device):
    model.eval()
    
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    
    return epoch_loss, epoch_acc

In [12]:
num_epochs = 10

best_acc = 0
patience = 2   
counter = 0

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    valid_loss, valid_acc = valid_one_epoch(model, valid_loader, criterion, device)

    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Valid Loss: {valid_loss:.4f} | Valid Acc: {valid_acc:.4f}")
    print("-" * 40)

    scheduler.step()

    # best model 저장
    if valid_acc > best_acc:
        best_acc = valid_acc
        counter = 0
        torch.save(model.state_dict(), "best_model.pth")
    else:
        counter += 1

    # early stopping
    if counter >= patience:
        print("Early Stopping 발생")
        break

Epoch [1/10]
Train Loss: 0.7932 | Train Acc: 0.7070
Valid Loss: 0.6748 | Valid Acc: 0.7584
----------------------------------------
Epoch [2/10]
Train Loss: 0.6327 | Train Acc: 0.7690
Valid Loss: 0.6693 | Valid Acc: 0.7610
----------------------------------------
Epoch [3/10]
Train Loss: 0.5711 | Train Acc: 0.7969
Valid Loss: 0.6986 | Valid Acc: 0.7416
----------------------------------------
Epoch [4/10]
Train Loss: 0.5198 | Train Acc: 0.8174
Valid Loss: 0.6460 | Valid Acc: 0.7626
----------------------------------------
Epoch [5/10]
Train Loss: 0.4837 | Train Acc: 0.8276
Valid Loss: 0.5040 | Valid Acc: 0.8245
----------------------------------------
Epoch [6/10]
Train Loss: 0.4410 | Train Acc: 0.8459
Valid Loss: 0.5343 | Valid Acc: 0.8136
----------------------------------------
Epoch [7/10]
Train Loss: 0.3982 | Train Acc: 0.8596
Valid Loss: 0.5250 | Valid Acc: 0.8079
----------------------------------------
Early Stopping 발생
